# Patron de Reflexion: un agente que se autocritica

Este notebook muestra el patron de **reflexion** (generar -> criticar -> revisar): un LLM produce una respuesta, otro rol del mismo modelo (el "revisor") la critica, y si encuentra un problema el modelo la corrige. Se repite hasta que el revisor la aprueba o se agotan los intentos.

Caso de uso: un agente que explica conceptos de fisica a un estudiante, y se autocorrige antes de entregar la respuesta final.

Las dos "voces" (explicador y revisor) son el mismo LLM con distintas instrucciones de sistema.

Requisitos: Ollama corriendo localmente con al menos el modelo `llama3.2:1b` descargado (ver `ayuda.txt`).

In [1]:
from reflection_agent import explicar, revisar, corregir, run_reflection
from display_helpers import show_trace

## 1. Sin reflexion (linea base)

Primero pedimos la explicacion una sola vez, sin ningun paso de revision. Con un modelo pequeno como `llama3.2:1b`, a veces la explicacion tiene errores o imprecisiones que pasan sin que nadie las revise.

In [5]:
concepto = "por que el periodo de un pendulo simple no depende de la masa"
model = "llama3.2:1b"

explicacion_base = explicar(concepto, model)
print(explicacion_base)

Excelente pregunta! En física, es común pensar que el período de un pendúlo dependa de la masa del objeto que lo mantiene suspendido. Pero eso no es cierto.

El período se refiere a la duración del movimiento circular de un objeto en torno al centro de gravedad. Es una medida que depende tanto de la velocidad como de la masa, porque más rápido o lento debe moverse para realizar el mismo número de vueltas por segundo.

Sin embargo, cuanto más pesado sea el objeto, menor es su perímetro. Pero no significa que el período se multiplique por un factor determinado. Esto ocurre porque la velocidad necesaria para que el pendúlo se mueva en torno al centro de gravedad depende de su masa.

En otras palabras, si dos objetos con masas diferentes se mueven a través del mismo espacio y están en el mismo punto terrestre, el objeto más pesado no tendrá la misma velocidad que el objeto menos pesado para realizar lo mismo número de vueltas. Por lo tanto, sus períodos serán diferentes.

Así que, en resum

## 2. El ciclo paso a paso

El patron tiene tres funciones en `reflection_agent.py`:

- `explicar(concepto, model)` -- genera una primera explicacion.
- `revisar(concepto, explicacion, model)` -- el mismo modelo, con otra instruccion de sistema, actua como revisor: dice si la aprueba y por que no si la rechaza.
- `corregir(concepto, explicacion, comentarios, model)` -- vuelve a generar la explicacion, esta vez viendo los comentarios del revisor.

Vamos a correrlas a mano una vez, sobre la explicacion de la seccion anterior.

In [3]:
revision = revisar(concepto, explicacion_base, model)
print("Aprobada:", revision["aprobada"])
print(revision["comentarios"])

Aprobada: False
APOBADA: Sí

COMENTARIOS: La expresión "la energía cinética de las partículas" es una mala manera de describir el problema. El momento angular o el periodo no dependen de la masa, sino de la altura y la velocidad con que se mueve la pendúlvica. Además, si no consideramos la masa en el sistema total del pendulo simple, como cuando incluimos un objeto aislado que vibra sobre las dos pendúlvicas, entonces sí es más complicado calcular e interpretar los resultados.


In [4]:
if not revision["aprobada"]:
    explicacion_corregida = corregir(concepto, explicacion_base, revision["comentarios"], model)
    print(explicacion_corregida)
else:
    print("El revisor la aprobo, no hace falta corregir.")

Excelente revisión! Me alegra saber que mi explicación no era lo suficientemente precisa. A continuación, te proporciono la versión corregida:

APOBADA: Sí

COMENTARIOS: La expresión "la energía cinética de las partículas" es una mala manera de describir el problema. El momento angular o el periodo no dependen de la masa, sino de la altura y la velocidad con que se mueve la pendúlvica. Además, cuando consideramos un sistema simple como un pendulo simple con dos pendúlvicas opuestas, los factores que influyen en su período son diferentes: la masa del objeto suspendido, las constantes físicas de la gravedad y las propiedades físicas del material de las pendúlvicas. Para calcular el período de este sistema, es necesario considerar estas variables para obtener una representación más precisa.

En otras palabras, la relación entre el momento angular y el período no depende de la masa individual, sino de la cantidad total de materia en el sistema (masa del objeto suspendido + materia del pend

## 3. El ciclo completo: `run_reflection`

`run_reflection` encadena estos tres pasos automaticamente hasta `max_rounds` veces, o hasta que el revisor aprueba. Devuelve la traza completa (cada ronda con su explicacion y su revision) para poder inspeccionar que paso.

In [6]:
trace = run_reflection(concepto, model=model, max_rounds=3)
show_trace(concepto, trace)

## Concepto: por que el periodo de un pendulo simple no depende de la masa

### Ronda 0

Disculpa el error anterior. Aquí te explico:

El período de un pendúlo simple es una función del acelerómetro, y no está relacionada con la masa del pendúlo en sí. La razón por la que no depende de la masa es que, aunque el pendúlo puede variar su peso y gravedad a lo largo de su vida, su resistencia a cambiar su movimiento circular permanece constante.

En otras palabras, la masa del pendúlo no influye en su periodo; solo sus características físicas. La razón por la que un pendúlo con la misma massa puede tener un período diferente depende de su acelerómetro, que mide la variable física que está relacionada con el movimiento circular del pendúlo.

En otras palabras, si tienes dos pendulos similares pero con masses diferentes y colgados en la misma altura, uno de ellos tendrá un periodo menor porque se moverá más rápidamente. Pero hay muchos factores que pueden afectar su período, como la inclinación, el peso del pendúlo y las constantes físicas de la gravedad, lo que significa que no depende solo de la masa.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA
La explicación proporcionada es clara y concisa, pero podría ser más detallada en algunos puntos para asegurarse de que el estudiante entienda bien los conceptos.

COMENTARIOS:
En cuanto a la frase "la razón por la cual no depende de la masa", podríamos agregar un poco más sobre cómo se mide la masa del pendúlo. ¿Podría explicar que se mide mediante una función que relaciona la masa con la aceleración, y luego se utiliza para calcular el período? De esta manera, el estudiante entenderá mejor cómo funciona este concepto.

Además, podríamos mencionar más sobre las constantes físicas de la gravedad que afectan la masa del pendúlo. Esto podría ayudar a asegurar que el estudiante tenga una comprensión más profunda de los fundamentos detrás del concepto.

### Ronda 1

...

### Ronda 2

Entiendo perfectamente. Gracias por señalar mi error. Aquí te presento la versión corregida:

El período de un pendúlo simple es una función que depende de la masa del pendúlo en sí, no del acelerómetro. Esto se debe a las constantes físicas que determinan cómo funciona el sistema.

La razón principal por la cual esto es así está relacionada con la gravedad y su relación con la masa:

g = m / r

donde r es el radio del pendúlo.

El período (T) del pendúlo también depende de las constantes físicas de la gravedad, como la constante de Gravidad (G), la masa (m) del pendúlo y su velocidad angular (ω).

En otras palabras, si tienes dos pendulos similares pero con masses diferentes y colgados en la misma altura, uno de ellos tendrá un período menor porque se moverá más rápidamente. Pero hay muchos factores que pueden afectar su período, como la inclinación, el peso del pendúlo y las constantes físicas de la gravedad.

Espero que esta versión corregida sea más precisa y clara para tu estudiante.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA
Ninguno

### Ronda 3

Gracias por señalar mi error. Aquí te presento la versión corregida:

El período de un pendúlo simple es una función que depende de la masa del pendúlo en sí, no del acelerómetro.

La razón principal por la cual esto es así está relacionada con las constantes físicas que determinan cómo funciona el sistema. En particular, se deben considerar dos factores clave:

1.  La masa (m) del pendúlo: El tiempo que toma un pendúlo para completar una órbita alrededor de su propio eje es directamente proporcional a la masa.
2.  Las constantes físicas de la gravedad, como la constante de Gravidad (G), la velocidad angular (ω) y las distancias entre el centro del eje y los puntos más cercanos:

## 4. Varios conceptos

Probemos con un par de conceptos mas -- el codigo no cambia, solo el texto que le pasamos.

In [7]:
conceptos = [
    "por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo",
    "diferencia entre velocidad y aceleracion",
]

for c in conceptos:
    trace = run_reflection(c, model=model, max_rounds=3)
    show_trace(c, trace)

## Concepto: por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo

### Ronda 0

¡Claro! La razón detrás de este fenómeno es bastante simple. Cuando un objeto cae hacia abajo, sus velocidades disminuyen con el tiempo debido a la acceleración constante. Sin embargo, cuando un objeto se lanza horizontalmente desde la misma altura que un objeto en caída libre, hay una especie de "sustitución" en su movimiento.

La velocidad vertical del objeto lanzado (horizontal) es menor en comparación con la caída libre, mientras que la velocidad vertical del objeto en caída libre aumenta. Esto significa que el objeto en caída libre disminuye en velocidad vertical más rápido que el objeto lanzado horizontalmente.

En este caso, los dos objetos tocan el suelo al mismo tiempo porque tienen velocidades verticales muy diferentes. El objeto en caída libre se acerca más a la altura del suelo con cada segundo que pasa, mientras que el objeto lanzado horizontalmente sigue aumentando en velocidad vertical. Esto significa que, aunque el objeto en caída libre disminuye más rápido en velocidad vertical, aún tiene suficiente energía para alcanzar la altura inicial.

En resumen, la velocidad de las velocidades verticales es lo que determina el momento en que tocan el suelo: si hay una mayor diferencia entre ambas velocidades o se aportan velocidades verticales paralelas.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA
No hay ningún error conceptual ni imprecisiones clarificativas en la explicación del profesor. La respuesta es clara y concisa, con un buen resumen de las bases físicas subyacentes al fenómeno descrito. Sin embargo, podríamos agregar una breve disculpa por el uso de la palabra "sustitución" para describir el movimiento de los objetos en caída libre y lanzado horizontalmente, ya que puede interpretarse como un cambio gradual en su movimiento instead de una sustitución real del modelo.

Ninguno

### Ronda 1

...

¡Claro! En física, dos conceptos importantes son velocidad y aceleración. Aunque pueden parecer similares, tienen significados distintos.

La **velocidad** se refiere al movimiento de un objeto en una dirección, ya sea horizontal o vertical. Es la distancia recorrida por el objeto en un período de tiempo determinado, medida en kilómetros por hora, metros por segundo u otras unidades. En otras palabras, es la cantidad de tiempo que tarda un objeto en cubrir una distancia específica.

Por otro lado, la **aceleración** es la relación entre la fuerza aplicada a un objeto y su masa o peso. Es el cambio en la velocidad de un objeto con el tiempo, medida en metros por segundo cuadrado por segundo u otras unidades. En otras palabras, es la cantidad de velocidad que se aumenta o disminuye en un período de tiempo determinado.

Por ejemplo, si una persona camina a un ritmo constante de 5 kilómetros por hora, su velocidad no cambia, ya que está moviéndose a la misma velocidad. Sin embargo, si esta persona recibe un golpe en el pecho, su aceleración será mayor porque su fuerza muscular debe trabajar más para mantener su velocidad.

En resumen, la velocidad se refiere al movimiento de un objeto, mientras que la aceleración es la relación entre la fuerza aplicada y la masa o peso del objeto.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA
La explicacion proporcionada tiene algunos errores conceptuales. Por ejemplo, en la definición de velocidad como la distancia recorrida por el objeto en un período de tiempo determinado, se utiliza términos incorrectos ("kilómetros por hora", "metros por segundo u otras unidades" en lugar de "kilómetros por hora", "métros por segundo", etc.). Además, la explicación sobre las diferencias entre velocidad y aceleración no es clara para un estudiante no experimentado con física.

### Ronda 1

Me disculpo por los errores cometidos en mi respuesta anterior. A continuación, te presento una versión corregida del texto:

Explicando la diferencia entre velocidad y aceleración

En física, dos conceptos importantes son velocidad y aceleración. La **velocidad** se refiere al movimiento de un objeto en una dirección, ya sea horizontal o vertical. Es el cambio en la distancia recorrida por el objeto con respecto a su posición original. Por ejemplo, si un carro viaja a 30 kilómetros por hora, su velocidad es constante.

Por otro lado, la **aceleración** es la relación entre la fuerza aplicada a un objeto y su masa o peso. Es el cambio en la dirección de movimiento de un objeto con respecto a su posición original. Por ejemplo, si una persona recibe un golpe en el pecho, su aceleración será mayor porque su fuerza muscular debe trabajar más para cambiar su dirección.

Es importante destacar que la velocidad se mide en kilómetros por hora o metros por segundo, mientras que la aceleración se mide en grados por segundo al cuadrado (g/s²). Además, la aceleración también puede ser considerada como una variable que implica cambios en la dirección de movimiento.

En resumen, la velocidad se refiere a la distancia recorrida por un objeto, mientras que la aceleración es la relación entre la fuerza aplicada y el aumento o disminución de su velocidad.

**Revision del critico -- Aprobada**

APROBADA: Si bien tu respuesta proporciona una explicación clara sobre las diferencias entre velocidad y aceleración, hay un punto de mejora importante.

COMENTARIOS:
La frase "por ejemplo, si un carro viaja a 30 kilómetros por hora, su velocidad es constante" podría ser más precisa. La velocidad no se mide necesariamente en kilómetros por hora o metros por segundo, sino que se puede medir en unidades específicas del sistema de medida (km/h, m/s, etc.). Además, la velocidad se refiere a la distancia recorrida por el tiempo, mientras que la velocidad es un valor absoluto. Por lo tanto, si el carro viaja 30 km en 1 hora, su velocidad absoluta es 30 km/h.

Ninguno

## 5. Modelo pequeno vs modelo mas grande

`llama3.2:1b` es rapido pero poco confiable, y eso se nota **en los dos roles**: el explicador puede cometer errores, y el revisor puede no detectarlos (o rechazar explicaciones que si estaban bien). Un modelo mas grande, como `qwen3:1.7b`, suele ser mas consistente en ambos roles -- sin tocar el resto del codigo, solo cambia el parametro `model`.

In [ ]:
model_grande = "qwen3:1.7b"
trace = run_reflection(concepto, model=model_grande, max_rounds=3)
show_trace(concepto, trace)


La idea central -- **un LLM que critica y corrige su propia salida antes de entregarla** -- se traslada a otros flujos de un curso o laboratorio:

- **Retroalimentacion de tareas**: el agente redacta un comentario sobre la respuesta de un estudiante, y un segundo paso revisa que la retroalimentacion sea justa y este bien fundamentada antes de mostrarsela.
- **Otros conceptos**: solo cambia el texto que le pasas a `run_reflection`, no hace falta tocar el codigo.
- **Otro criterio de revision**: cambia `REVISOR_PROMPT` en `reflection_agent.py` para que el revisor chequee otra cosa (ej. que la explicacion use el vocabulario correcto para el nivel del curso, o que incluya una analogia).
- **Combinarlo con herramientas**: se puede mezclar con el patron de `Tools` -- por ejemplo, un agente que ajusta una curva y *despues* reflexiona sobre si el modelo elegido tiene sentido fisico.